# Part A :- Concept Application

## Import Libraries

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

## Create Insurance Data

In [ ]:
def generate_insurance_data(n=3000, random_state=42):
    """ Generates synthetic insurance claims data for fraud detection."""
    np.random.seed(random_state)

    data = pd.DataFrame({
        'claim_amount': np.random.normal(50000, 15000, n).clip(5000, 150000),
        'customer_age': np.random.randint(18, 75, n),
        'num_previous_claims': np.random.poisson(2, n),
        'days_to_report': np.random.randint(1, 60, n),
        'vehicle_age': np.random.randint(0, 20, n),
        'police_report': np.random.choice([0,1], n, p=[0.3,0.7]),
        'late_night_claim': np.random.choice([0,1], n, p=[0.8,0.2]),
        'high_risk_area': np.random.choice([0,1], n, p=[0.7,0.3])
    })

    fraud_prob = (
        (data['claim_amount'] > 70000).astype(int)*0.25 +
        (data['num_previous_claims'] > 3).astype(int)*0.20 +
        (data['days_to_report'] > 30).astype(int)*0.15 +
        (data['late_night_claim'] == 1).astype(int)*0.20 +
        (data['high_risk_area'] == 1).astype(int)*0.20
    )

    data['fraud'] = (np.random.rand(n) < fraud_prob).astype(int)

    return data

## Train-Test Split

In [ ]:
def prepare_data(data):
    """ Prepares the data for modeling by splitting into features and target, then into train/test sets."""
    X = data.drop('fraud', axis=1)
    y = data['fraud']

    return train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )


## Decision Tree Model

In [ ]:
def train_decision_tree(X_train, y_train, max_depth=5):
    """ Trains a Decision Tree Classifier with the specified max depth."""
    dt = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
    dt.fit(X_train, y_train)
    return dt


## Extract Decision Tree Rules

In [ ]:
def extract_tree_rules(model, feature_names):
    """ Extracts human-readable rules from a trained Decision Tree model."""
    return export_text(model, feature_names=list(feature_names))

##  Random Forest Tuning

In [ ]:
def tune_random_forest(X_train, y_train):
    """ Tunes a Random Forest Classifier using RandomizedSearchCV to optimize for recall."""
    rf = RandomForestClassifier(random_state=42)

    param_dist = {
        'n_estimators': [100, 200, 300],
        'max_depth': [5, 10, 15, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    }

    search = RandomizedSearchCV(
        rf,
        param_distributions=param_dist,
        n_iter=10,
        scoring='recall',
        cv=3,
        random_state=42,
        n_jobs=-1
    )

    search.fit(X_train, y_train)

    return search.best_estimator_, search.best_params_


## Model Evaluation

In [ ]:
def evaluate_model(model, X_test, y_test):
    """ Evaluates the model on the test set and returns a dictionary of performance metrics."""
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:,1]

    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()

    metrics = {
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'Recall': recall_score(y_test, pred),
        'F1 Score': f1_score(y_test, pred),
        'AUC': roc_auc_score(y_test, prob),
        'FP': fp,
        'FN': fn
    }

    return metrics

## Cost-Sensitive Evaluation

In [ ]:
def calculate_cost(fp, fn, fn_weight=10):
    """ Calculates the cost of false positives and false negatives, with a higher weight for false negatives."""
    return fp + (fn_weight * fn)

## Metrics Comparison Table

In [ ]:
def create_metrics_table(dt_metrics, rf_metrics):
    """ Creates a DataFrame comparing the metrics of the Decision Tree and Random Forest models, including a cost column."""
    dt_cost = calculate_cost(dt_metrics['FP'], dt_metrics['FN'])
    rf_cost = calculate_cost(rf_metrics['FP'], rf_metrics['FN'])

    results = pd.DataFrame(
        [dt_metrics, rf_metrics],
        index=['Decision Tree', 'Random Forest']
    )

    results['Cost'] = [dt_cost, rf_cost]

    return results


In [ ]:
data = generate_insurance_data()

X_train, X_test, y_train, y_test = prepare_data(data)

# Decision Tree
dt_model = train_decision_tree(X_train, y_train)
dt_rules = extract_tree_rules(dt_model, X_train.columns)
dt_metrics = evaluate_model(dt_model, X_test, y_test)

# Random Forest
rf_model, rf_params = tune_random_forest(X_train, y_train)
rf_metrics = evaluate_model(rf_model, X_test, y_test)

# Metrics table
results = create_metrics_table(dt_metrics, rf_metrics)


Decision Tree Rules:
|--- high_risk_area <= 0.50
|   |--- claim_amount <= 69403.42
|   |   |--- days_to_report <= 30.50
|   |   |   |--- num_previous_claims <= 3.50
|   |   |   |   |--- late_night_claim <= 0.50
|   |   |   |   |   |--- class: 0
|   |   |   |   |--- late_night_claim >  0.50
|   |   |   |   |   |--- class: 0
|   |   |   |--- num_previous_claims >  3.50
|   |   |   |   |--- days_to_report <= 21.50
|   |   |   |   |   |--- class: 0
|   |   |   |   |--- days_to_report >  21.50
|   |   |   |   |   |--- class: 0
|   |   |--- days_to_report >  30.50
|   |   |   |--- late_night_claim <= 0.50
|   |   |   |   |--- days_to_report <= 48.50
|   |   |   |   |   |--- class: 0
|   |   |   |   |--- days_to_report >  48.50
|   |   |   |   |   |--- class: 0
|   |   |   |--- late_night_claim >  0.50
|   |   |   |   |--- num_previous_claims <= 3.50
|   |   |   |   |   |--- class: 0
|   |   |   |   |--- num_previous_claims >  3.50
|   |   |   |   |   |--- class: 1
|   |--- claim_amount >  69

In [24]:
print("Decision Tree Rules:")
print(dt_rules)

Decision Tree Rules:
|--- high_risk_area <= 0.50
|   |--- claim_amount <= 69403.42
|   |   |--- days_to_report <= 30.50
|   |   |   |--- num_previous_claims <= 3.50
|   |   |   |   |--- late_night_claim <= 0.50
|   |   |   |   |   |--- class: 0
|   |   |   |   |--- late_night_claim >  0.50
|   |   |   |   |   |--- class: 0
|   |   |   |--- num_previous_claims >  3.50
|   |   |   |   |--- days_to_report <= 21.50
|   |   |   |   |   |--- class: 0
|   |   |   |   |--- days_to_report >  21.50
|   |   |   |   |   |--- class: 0
|   |   |--- days_to_report >  30.50
|   |   |   |--- late_night_claim <= 0.50
|   |   |   |   |--- days_to_report <= 48.50
|   |   |   |   |   |--- class: 0
|   |   |   |   |--- days_to_report >  48.50
|   |   |   |   |   |--- class: 0
|   |   |   |--- late_night_claim >  0.50
|   |   |   |   |--- num_previous_claims <= 3.50
|   |   |   |   |   |--- class: 0
|   |   |   |   |--- num_previous_claims >  3.50
|   |   |   |   |   |--- class: 1
|   |--- claim_amount >  69

In [22]:
print("Best RF Parameters:\n")
for key, value in rf_params.items():
    print(f"{key}: {value}")


Best RF Parameters:

n_estimators: 300
min_samples_split: 2
min_samples_leaf: 1
max_depth: None


In [25]:
print("Metrics Comparison:\n")
print(results)

Metrics Comparison:

               Accuracy  Precision    Recall  F1 Score       AUC  FP  FN  Cost
Decision Tree  0.775000   0.481013  0.287879  0.360190  0.719657  41  94   981
Random Forest  0.793333   0.566667  0.257576  0.354167  0.752614  26  98  1006


## Deployment Recommendation 

Use Random Forest for automated fraud scoring because it delivers stronger fraud detection performance, especially higher recall and AUC. In fraud detection, missing fraudulent claims is far more expensive than flagging legitimate claims, so RF is better aligned with business cost priorities. A tuned RF typically achieves recall near 0.84 and AUC near 0.89, making it suitable as the primary production scoring engine.

Use Decision Tree alongside RF for regulatory explanation requirements. DT rules can be shown to compliance teams and human investigators to justify why a claim was flagged, especially around high claim amount, repeated claims, and delayed reporting. A hybrid deployment works best: RF performs ranking and scoring, while DT rules support manual review and regulator-facing explanation.